# Programme 5 — Newton scalaire

**Référence :** Kröger, *Numerische Mathematik 1 für AMP*, chapitre 3.1.

Ce notebook accompagne `newton_scalar.py` et illustre :

1. La formule de Newton (3.3) et la preuve de convergence quadratique (Satz 3.5)
2. La reproduction de l'**Übung 3.1** : $f(x) = x^2 - 3$
3. L'**ordre de convergence expérimental** (section 3.1.4)
4. Les **critères d'arrêt** combinés (section 3.1.5)
5. La comparaison Newton / Sécante / Bissection
6. Le **Newton modifié** pour les nullstelles multiples (section 3.1.8)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from newton_scalar import (
    newton, secante, bissection, newton_modifie,
    ordre_experimental_exact, ordre_experimental_sans_solution,
    condition_nullstelle,
    tracer_convergence, tracer_newton_graphique, tracer_ordres_experimentaux,
)

## 1. Le principe (Figure 3.2 du script)

La formule de Newton :
$$x^+ = x - \frac{f(x)}{f'(x)}$$

Géométriquement : on remplace $f$ par sa tangente en $x$, et on prend le zéro de cette tangente comme nouvelle approximation.

In [ ]:
f = lambda x: x**2 - 3
df = lambda x: 2*x

res = newton(f, df, 4.5)
tracer_newton_graphique(f, res.historique_x, intervalle=(1, 5), n_steps=4)
plt.show()

## 2. Übung 3.1 : $f(x) = x^2 - 3$, $x_0 = 4.5$

Solution exacte : $x^* = \sqrt{3} \approx 1.7320508...$

In [ ]:
x_star = np.sqrt(3)

print(f'{"k":>3} | {"x_k":>18} | {"f(x_k)":>14} | {"|x_k - x*|":>14}')
print('-' * 58)
for k, xi in enumerate(res.historique_x):
    print(f'{k:>3} | {xi:>18.15f} | {f(xi):>14.6e} | {abs(xi - x_star):>14.6e}')

À chaque itération, le nombre de chiffres corrects **double** environ — c'est la signature de la convergence quadratique (Satz 3.5).

## 3. Mesure de l'ordre de convergence (section 3.1.4)

Le script donne deux formules pour estimer l'ordre $\alpha$ :

**Avec $x^*$ connu :** $\alpha \approx \frac{\ln(e_k / e_{k+1})}{\ln(e_{k-1} / e_k)}$ où $e_k = |x_k - x^*|$.

**Sans $x^*$ :** même formule avec $d_k = |x_k - x_{k+1}|$ à la place de $e_k$. Valide pour $k$ assez grand (section 3.1.4, formule 3.10).

In [ ]:
ordres = ordre_experimental_exact(res.historique_x, x_star)
ordres_sans = ordre_experimental_sans_solution(res.historique_x)

print(f'{"k":>3} | {"α (x* connu)":>14} | {"α (sans x*)":>14}')
print('-' * 38)
for i, (o1, o2) in enumerate(zip(ordres, ordres_sans)):
    print(f'{i+1:>3} | {o1:>14.6f} | {o2:>14.6f}')

print(f'\n→ Converge vers α = 2 (quadratique) ✓')

## 4. Comparaison Newton / Sécante / Bissection

On compare les trois méthodes sur $f(x) = x^3 - 2x - 5$ (racine $x^* \approx 2.0946$).

| Méthode | Ordre de convergence | Évaluations de $f$ par step | Besoin de $f'$ ? |
|---|---|---|---|
| Newton | 2 (quadratique) | $f$ + $f'$ | oui |
| Sécante | $\phi \approx 1.618$ | 1 seule $f$ | non |
| Bissection | 1 (linéaire) | 1 seule $f$ | non |

In [ ]:
f2 = lambda x: x**3 - 2*x - 5
df2 = lambda x: 3*x**2 - 2
x_star2 = 2.0945514815423265

res_n = newton(f2, df2, 3.0)
res_s = secante(f2, 2.0, 3.0)
res_b = bissection(f2, 2.0, 3.0)

for r in [res_n, res_s, res_b]:
    print(r)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
tracer_convergence([res_n, res_s, res_b], x_star=x_star2, ax=axes[0])
tracer_ordres_experimentaux([res_n, res_s], x_star=x_star2, ax=axes[1])
plt.tight_layout()
plt.show()

**Lecture du graphique de gauche :** Newton descend le plus vite (pente la plus raide). Sécante est légèrement plus lent mais n'a pas besoin de $f'$. Bissection : lent et régulier, un bit de précision par itération.

**Graphique de droite :** l'ordre expérimental de Newton stabilise à $\alpha \approx 2$, celui de la sécante à $\alpha \approx 1.618$ (le nombre d'or, issu de $\alpha^2 = \alpha + 1$).

## 5. Condition du problème (section 3.1.2)

$$\text{cond}_{abs} = \frac{1}{|f'(x^*)|}.$$

Si $f'(x^*) \approx 0$ (nullstelle multiple ou quasi-multiple), le problème est **mal conditionné** : même une petite perturbation de $f$ déplace beaucoup la racine.

In [ ]:
# Nullstelle simple bien conditionnée
print(f'f(x) = x² - 3 :  f\'(√3) = {df(x_star):.6f}, cond = {condition_nullstelle(df(x_star)):.4f}')

# Nullstelle où f' est petit → mal conditionné
g = lambda x: (x - 1)**2 - 0.01  # racines à 0.9 et 1.1, f'(x*) = ±0.2
dg = lambda x: 2*(x - 1)
print(f'g(x) = (x-1)² - 0.01 : g\'(1.1) = {dg(1.1):.6f}, cond = {condition_nullstelle(dg(1.1)):.4f}')

# Nullstelle multiple → très mal conditionné
print(f'h(x) = x³ :           h\'(0) = 0,      cond = ∞  (3-fache Nullstelle)')

## 6. Newton modifié pour nullstelles multiples (section 3.1.8)

Si $x^*$ est une nullstelle de multiplicité $m$, Newton standard ne converge que **linéairement** (perte de la convergence quadratique).

**Solution :** Newton modifié $x^+ = x - m \cdot \frac{f(x)}{f'(x)}$. Retrouve la convergence quadratique si $m$ est correctement choisi.

In [ ]:
f3 = lambda x: x**3  # triple Nullstelle en 0
df3 = lambda x: 3*x**2

res_std = newton(f3, df3, 1.0, tol=1e-10)
res_m2 = newton_modifie(f3, df3, 1.0, multiplicite=2, tol=1e-10)
res_m3 = newton_modifie(f3, df3, 1.0, multiplicite=3, tol=1e-10)

print(f'Newton standard  (m=1) : {res_std.iterations:>3} itérations')
print(f'Newton modifié   (m=2) : {res_m2.iterations:>3} itérations')
print(f'Newton modifié   (m=3) : {res_m3.iterations:>3} itérations  ← retrouve α = 2 !')

In [ ]:
tracer_convergence([res_std, res_m2, res_m3], x_star=0.0)
plt.title('Nullstelle triple : Newton standard vs modifié')
plt.show()

## Récapitulatif

| Méthode | Ordre | Coût/step | Robustesse | Besoin |
|---|---|---|---|---|
| Newton | 2 | $f + f'$ | locale | $f'$ explicite |
| Sécante | $\phi \approx 1.618$ | $f$ seul | locale | 2 points initiaux |
| Bissection | 1 | $f$ seul | **globale** | changement de signe |
| Newton modifié | 2 (si m correct) | $f + f'$ | locale | multiplicité connue |

**Stratégie hybride (section 3.1.6) :** commencer par la bissection (globalement convergent), puis basculer vers Newton une fois dans le basin d'attraction.

**Prochain programme :** `runge_phenomenon.py` — interpolation polynomiale et le célèbre phénomène de Runge (chapitre 4).